> ## SOLUTION / ANSWER KEY &mdash; Lab 10.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-debug-and-fix.ipynb`](../lab-10-debug-and-fix.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.10 &mdash; Debug & Fix the Loop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Run a broken agent and inspect its intermediate steps
- Diagnose the wrong-tool bug from the trace
- Apply the fix (a grounding tool) and verify it improved

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Modules 6&ndash;9. Advanced labs end with an **optional** cell that runs the **real** library. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Debugging, in code](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-10-10"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
The full debug loop (deck slides 14&ndash;16): run the agent with the trace visible, **read** it to
localise the bug, **diagnose** the failure mode, **fix** at the right layer, and **verify** with a re-run.
Here a buggy agent calls a tool it wasn't given (wrong tool) and then computes on an **ungrounded** number;
the fix is to give it a **grounding** tool and confirm the output is now grounded and correct.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into a ReAct agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}
def parse_react(text):
    """Turn the model's ReAct text into ('final', answer) or ('action', name, input)."""
    action = inp = None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("Final Answer:"): return ("final", s.split(":", 1)[1].strip())
        if s.startswith("Action Input:"): inp = s.split(":", 1)[1].strip()
        elif s.startswith("Action:"):      action = s.split(":", 1)[1].strip()
    return ("action", action, inp)
class AgentExecutor:
    """Runs the loop: ask model -> parse -> run tool -> observe -> repeat, capped by max_iterations
    (mirrors langchain.agents.AgentExecutor). verbose=True prints the ReAct trace -- your #1 debug tool."""
    def __init__(self, agent, tools=None, verbose=False, max_iterations=6):
        self.agent = agent
        self.tools = agent["tools"] if tools is None else {t.name: t for t in tools}
        self.model = agent["model"]; self.prompt = agent["prompt"]
        self.verbose = verbose; self.max_iterations = max_iterations
    def invoke(self, inputs):
        scratch, steps = "", []
        for _ in range(self.max_iterations):
            text = self.model.invoke(self.prompt.format(input=inputs["input"], scratchpad=scratch)).content
            if self.verbose: print(text)
            parsed = parse_react(text)
            if parsed[0] == "final":
                return {"output": parsed[1], "intermediate_steps": steps}
            name, arg = parsed[1], parsed[2]
            obs = self.tools[name].invoke(arg) if name in self.tools else ("unknown tool: %s" % name)
            if self.verbose: print("Observation:", obs)
            steps.append((name, arg, obs)); scratch += text + "\nObservation: " + str(obs) + "\n"
        return {"output": None, "intermediate_steps": steps}

@tool
def extract_figure(name):
    """Ground a figure with its source from the filing."""
    return {"revenue": {"value": 120.0, "source": "p4"}}.get(name, {})
@tool
def compute(expression):
    """Exact arithmetic."""
    return safe_calc(expression)

BUGGY_SCRIPT = [
    "Thought: I need revenue.\nAction: lookup_order\nAction Input: revenue",       # wrong tool -> unknown
    "Thought: I'll estimate.\nAction: compute\nAction Input: 0.15 * 100",           # 100 ungrounded
    "Thought: done.\nFinal Answer: ~15M (ungrounded)",
]
FIXED_SCRIPT = [
    "Thought: ground revenue first.\nAction: extract_figure\nAction Input: revenue",
    "Thought: 15% of the real 120.\nAction: compute\nAction Input: 0.15 * 120",
    "Thought: done.\nFinal Answer: 18.0M [p4]",
]
def run(script, tools):
    model = FakeChatModel(script)
    prompt = PromptTemplate.from_template("Q: {input}\n{scratchpad}")
    return AgentExecutor(create_react_agent(model, tools, prompt), max_iterations=6).invoke({"input": "15% of revenue?"})
print("buggy & fixed scripts ready")

## Your Turn
Complete `diagnose` (find the failure mode) and give the fixed agent its **grounding** tool.

In [ ]:
def diagnose(steps):
    # read the trace: what went wrong first?
    for tool, arg, obs in steps:
        if "unknown tool" in str(obs):
            return "wrong_tool"
    return "ok"

def run_buggy():
    return run(BUGGY_SCRIPT, [compute])          # note: extract_figure was NOT given

def run_fixed():
    return run(FIXED_SCRIPT, [extract_figure, compute])

try:
    b = run_buggy()
    print('buggy tools :', [s[0] for s in b['intermediate_steps']])
    print('diagnosis   :', diagnose(b['intermediate_steps']))
    print('buggy output:', b['output'])
    f = run_fixed()
    print('fixed tools :', [s[0] for s in f['intermediate_steps']])
    print('fixed output:', f['output'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the buggy run is diagnosed as wrong_tool", lambda: diagnose(run_buggy()["intermediate_steps"]) == "wrong_tool")
expect_true("the buggy output is ungrounded (no citation)", lambda: "[p" not in run_buggy()["output"])
expect_true("the fixed agent grounds first via extract_figure", lambda: run_fixed()["intermediate_steps"][0][0] == "extract_figure")
expect_true("the fixed output is grounded (cites p4)", lambda: "[p4]" in run_fixed()["output"])
expect_true("the fix actually changed the result", lambda: run_fixed()["output"] != run_buggy()["output"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
See the real debugging move: verbose=True prints the full trace so you can read it like a transcript. Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`. If a package, model or key is
missing the cell prints a friendly note and moves on.
**The graded steps above are offline &amp; deterministic, so the lab always verifies without a model.**

In [ ]:
try:
    from langchain.agents import AgentExecutor as RealAgentExecutor
    print("Real LangChain: AgentExecutor(agent=..., tools=..., verbose=True) prints every thought/action/observation.")
    print("Inspect result['intermediate_steps'] to assert exactly which tools ran, in what order, with what args --")
    print("that's how a regression test catches the wrong-tool bug. Add a Langfuse/LangSmith callback for prod traces.")
except Exception as e:
    print("Install langchain to see the real AgentExecutor -- skipping:", type(e).__name__)
print("The offline debug loop above already diagnosed the bug and verified the grounded fix.")

---
### Key takeaway
Run -> read the trace -> diagnose -> fix at the right layer -> verify. The buggy agent called a tool it lacked and computed on an ungrounded number; giving it a grounding tool fixed both, and the re-run proved it. That's the debug loop.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>